# 00a — Compute context utility \(U(k)\) from raw text

This notebook is the reference implementation of the paper's **context utility** measurement. It does **not** load precomputed \(C\) features. Given raw text, it computes

\[
L_k(x)=\frac{1}{|\mathcal T|}\sum_{t\in\mathcal T}
-\log q(x_t\mid x_{t-k:t-1}),
\qquad
U_x(k)=L_8(x)-L_k(x),
\]

using the frozen independent scorer `Qwen/Qwen3-0.6B-Base`.

Frozen measurement configuration used in the paper:

- scorer: `Qwen/Qwen3-0.6B-Base`
- scorer-token prefix: first 384 tokens
- context windows: `[8, 32, 128, 256]`
- target positions: 8 deterministic evenly spaced positions from token 256 through the end of the prefix
- batch size: 8 target positions
- no source-generator internals are used

The implementation below is cleaned from the actual project extractor rather than re-derived from the paper prose.

## Input format

Point `INPUT_PATH` at a `.parquet` or `.csv` table containing:

- one stable identifier column, configured by `ID_COL`;
- one raw-text column, configured by `TEXT_COL`.

The notebook writes a feature table containing the ID plus the four loss values and
`context_u_32`, `context_u_128`, and `context_u_256`.

For full-corpus reproduction, construct the input table from the public Ghostbuster/RAID source data and the exact manifests/splits released with this repository.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "experiments" else cwd
DATA = ROOT / "data"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# USER CONFIGURATION
# ------------------------------------------------------------------
# Example:
# INPUT_PATH = DATA / "raw" / "raid_pilot_texts.parquet"
INPUT_PATH = DATA / "raw" / "ghostbuster_detector_pool.parquet"
ID_COL = "row_id"
TEXT_COL = "text"

OUTPUT_PATH = RESULTS / "ghostbuster_detector_context.parquet"

# ------------------------------------------------------------------
# FROZEN PAPER CONFIGURATION
# ------------------------------------------------------------------
SCORER_NAME = "Qwen/Qwen3-0.6B-Base"
PREFIX_LEN = 384
CONTEXT_LENGTHS = [8, 32, 128, 256]
N_CONTEXT_TARGETS = 8
CONTEXT_BATCH_SIZE = 8

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("ROOT:", ROOT)
print("input:", INPUT_PATH)
print("output:", OUTPUT_PATH)
print("device:", DEVICE)

/home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROOT: ..
input: ../data/raw/ghostbuster_detector_pool.parquet
output: ../results/ghostbuster_detector_context.parquet
device: cuda


In [2]:
def load_table(path):
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported input type: {path.suffix}")

assert INPUT_PATH.exists(), (
    f"Missing {INPUT_PATH}. See the notebook header and data/README.md "
    "for how to construct a raw-text input table."
)

raw = load_table(INPUT_PATH)

missing = [c for c in [ID_COL, TEXT_COL] if c not in raw.columns]
assert not missing, f"Missing required columns: {missing}"
assert not raw[ID_COL].duplicated().any(), f"{ID_COL} must be unique."

print("rows:", len(raw))
display(raw[[ID_COL, TEXT_COL]].head())

rows: 9000


,row_id,text
0,original::wp::gpt::1,"Once upon a time, in the quaint village of Ser..."
1,original::wp::gpt::2,"Once upon a time, amidst the starry night sky,..."
2,original::wp::gpt::3,As I strolled through the dimly lit alleys of ...
3,original::wp::gpt::4,"Growing up, I was always a dreamer. My imagina..."
4,original::wp::gpt::5,"I jolted awake, my heart pounding against my c..."


## Load the independent scorer

The scorer is deliberately distinct from the source generators. The model is used only to evaluate the completed text.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(SCORER_NAME)

model = AutoModelForCausalLM.from_pretrained(
    SCORER_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)

model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████████████| 310/310 [00:00<00:00, 3109.11it/s]


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

## Reference implementation

A document is eligible only if the first 384 scorer tokens contain enough positions to evaluate all requested windows. Since the largest context is 256 tokens, every target position is at least 256 tokens into the scorer prefix.

The same target positions are used for **all** context lengths, so \(L_8,L_{32},L_{128},L_{256}\) differ only in the amount of preceding history shown to the scorer.

In [4]:
@torch.inference_mode()
def context_loss_curve(
    text,
    context_lengths=CONTEXT_LENGTHS,
    prefix_len=PREFIX_LEN,
    n_targets=N_CONTEXT_TARGETS,
    batch_size=CONTEXT_BATCH_SIZE,
):
    ids = tokenizer(
        str(text),
        add_special_tokens=False,
        return_tensors="pt",
    )["input_ids"][0]

    # Exact frozen prefix rule.
    ids = ids[:prefix_len]

    max_k = max(context_lengths)

    available_positions = np.arange(
        max_k,
        len(ids),
    )

    if len(available_positions) < n_targets:
        return np.full(
            len(context_lengths),
            np.nan,
            dtype=np.float32,
        )

    # Deterministic evenly spaced target positions.
    target_positions = np.linspace(
        max_k,
        len(ids) - 1,
        n_targets,
        dtype=int,
    )

    # Guard against accidental duplicates from integer rounding.
    target_positions = np.unique(
        target_positions
    )

    losses = []

    for k in context_lengths:
        sequences = []
        targets = []

        for pos in target_positions:
            sequences.append(
                ids[pos-k:pos]
            )
            targets.append(
                ids[pos]
            )

        total_nll = 0.0
        total_count = 0

        for start in range(
            0,
            len(sequences),
            batch_size,
        ):
            batch_sequences = sequences[
                start:start + batch_size
            ]
            batch_targets = targets[
                start:start + batch_size
            ]

            x = torch.stack(
                batch_sequences
            ).to(DEVICE)

            y = torch.stack(
                batch_targets
            ).to(DEVICE)

            # Qwen3 supports logits_to_keep=1. This was used in the
            # project implementation to avoid materializing unnecessary logits.
            out = model(
                input_ids=x,
                logits_to_keep=1,
            )

            logits = out.logits[:, -1, :]

            log_probs = torch.log_softmax(
                logits.float(),
                dim=-1,
            )

            nll = -log_probs[
                torch.arange(
                    len(y),
                    device=DEVICE,
                ),
                y,
            ]

            total_nll += nll.sum().item()
            total_count += len(y)

        losses.append(
            total_nll / total_count
        )

    return np.asarray(
        losses,
        dtype=np.float32,
    )


def context_features(text):
    losses = context_loss_curve(text)

    if not np.isfinite(losses).all():
        return {
            "context_loss_8": np.nan,
            "context_loss_32": np.nan,
            "context_loss_128": np.nan,
            "context_loss_256": np.nan,
            "context_u_32": np.nan,
            "context_u_128": np.nan,
            "context_u_256": np.nan,
        }

    l8, l32, l128, l256 = map(float, losses)

    return {
        "context_loss_8": l8,
        "context_loss_32": l32,
        "context_loss_128": l128,
        "context_loss_256": l256,
        "context_u_32": l8 - l32,
        "context_u_128": l8 - l128,
        "context_u_256": l8 - l256,
    }

## Sanity check on one document

This cell makes the measurement auditable at the single-document level before running the corpus.

In [5]:
example = raw.iloc[0]
example_features = context_features(example[TEXT_COL])

print("ID:", example[ID_COL])
print(example_features)

ID: original::wp::gpt::1
{'context_loss_8': 5.707314491271973, 'context_loss_32': 4.1928205490112305, 'context_loss_128': 3.3843488693237305, 'context_loss_256': 3.5767176151275635, 'context_u_32': 1.5144939422607422, 'context_u_128': 2.322965621948242, 'context_u_256': 2.130596876144409}


## Compute the full input table

The output is a cache of a deterministic transformation from raw text under the frozen scorer/configuration above. The cached table is convenient, but it is **not** the definition of the measurement.

In [6]:
rows = []

for row in tqdm(
    raw[[ID_COL, TEXT_COL]].itertuples(index=False, name=None),
    total=len(raw),
    desc="Context utility",
):
    row_id, text = row
    feat = context_features(text)

    rows.append({
        ID_COL: row_id,
        **feat,
    })

features = pd.DataFrame(rows)

features.to_parquet(
    OUTPUT_PATH,
    index=False,
)

print("saved:", OUTPUT_PATH)
print("shape:", features.shape)
display(features.head())
display(
    features[
        [
            "context_u_32",
            "context_u_128",
            "context_u_256",
        ]
    ].describe()
)

Context utility: 100%|██████████████████████████████| 9000/9000 [37:22<00:00,  4.01it/s]

saved: ../results/ghostbuster_detector_context.parquet
shape: (9000, 8)


,row_id,context_loss_8,context_loss_32,context_loss_128,context_loss_256,context_u_32,context_u_128,context_u_256
0,original::wp::gpt::1,5.707314,4.192821,3.384349,3.576718,1.514494,2.322966,2.130597
1,original::wp::gpt::2,2.608050,2.044599,2.262877,2.418936,0.563451,0.345173,0.189114
2,original::wp::gpt::3,2.704532,1.969186,1.427015,1.148244,0.735347,1.277517,1.556288
3,original::wp::gpt::4,2.720636,1.882901,1.310261,1.448591,0.837735,1.410375,1.272045
4,original::wp::gpt::5,3.873142,2.196480,1.876708,2.033973,1.676661,1.996434,1.839169


,context_u_32,context_u_128,context_u_256
count,8668.000000,8668.000000,8668.000000
mean,0.818480,1.269249,1.418647
std,0.587647,0.763334,0.809989
min,-2.060224,-1.179792,-0.962029
25%,0.412337,0.725222,0.840277
50%,0.750184,1.181051,1.326301
75%,1.162616,1.722096,1.904917
max,4.192546,5.855315,6.110774


## Optional cache verification

If this repository ships a frozen feature cache, point `REFERENCE_PATH` to it and configure the same stable ID. This checks that recomputation matches the released cache rather than asking the reader to trust the cache.

Small floating-point differences may occur across PyTorch/CUDA versions; they should be tiny relative to the reported effects.

In [8]:
REFERENCE_PATH = None
# Example:
# REFERENCE_PATH = DATA / "cached" / "raid_context_features.parquet"

if REFERENCE_PATH is not None:
    reference = pd.read_parquet(REFERENCE_PATH)

    compare_cols = [
        "context_u_32",
        "context_u_128",
        "context_u_256",
    ]

    chk = features.merge(
        reference[[ID_COL] + compare_cols],
        on=ID_COL,
        suffixes=("_recomputed", "_cached"),
        validate="one_to_one",
    )

    for c in compare_cols:
        err = np.abs(
            chk[f"{c}_recomputed"]
            - chk[f"{c}_cached"]
        )
        print(
            c,
            "n =", err.notna().sum(),
            "mean abs diff =", err.mean(),
            "max abs diff =", err.max(),
        )

In [9]:
raw = pd.read_parquet(
    DATA / "raw" / "ghostbuster_detector_pool.parquet"
)

feat = pd.read_parquet(
    RESULTS / "ghostbuster_detector_context.parquet"
)

check = raw.merge(
    feat,
    on="row_id",
    how="left",
    validate="one_to_one",
)

print("raw rows:", len(raw))
print("feature rows:", len(feat))
print("eligible:", check["context_u_256"].notna().sum())

display(
    check.groupby(["source", "domain", "generator"])[
        ["context_u_32", "context_u_128", "context_u_256"]
    ]
    .agg(["count", "mean"])
)

raw rows: 9000
feature rows: 9000
eligible: 8668


context_u_32           context_u_128            \
                                    count      mean         count      mean   
source      domain generator                                                  
claude_eval essay  claude            1000  0.893342          1000  1.331685   
            reuter claude            1000  0.864164          1000  1.316797   
            wp     claude             974  0.733406           974  1.155209   
original    essay  gpt               1000  0.919637          1000  1.465001   
                   human              997  0.651453           997  1.053557   
            reuter gpt                982  0.952241           982  1.485232   
                   human              984  0.898142           984  1.347159   
            wp     gpt                903  0.820752           903  1.288355   
                   human              828  0.596132           828  0.924287   

                             context_u_256            
                                     count      mean  
source      domain generator                          
claude_eval essay  claude             1000  1.524968  
            reuter claude             1000  1.493610  
            wp     claude              974  1.305381  
original    essay  gpt                1000  1.641615  
                   human               997  1.167251  
            reuter gpt                 982  1.667041  
                   human               984  1.474609  
            wp     gpt                 903  1.408964  
                   human               828  1.015824